# CNN Basics

Convolution, max/average pooling, and feature maps — implemented from scratch with full
backward passes, validated against PyTorch, and tested on sklearn digits.

We answer four concrete questions with numbers:
1. What does a 2D convolution actually compute, and how does backprop through it work?
2. How do pooling layers downsample and route gradients?
3. Why does parameter sharing make CNNs far more efficient than position-specific dense layers?
4. Does a CNN trained on 8×8 digits generalize better than a flattened MLP when digits are shifted?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. From-Scratch Implementations

**2D Convolution** slides a learnable filter bank over the input. For input $X$ of shape
$(N, C_{in}, H, W)$ and filters $W$ of shape $(C_{out}, C_{in}, k_H, k_W)$:

$$Y[n, c, i, j] = b_c + \sum_{c'=0}^{C_{in}-1}\sum_{u=0}^{k_H-1}\sum_{v=0}^{k_W-1}
W[c, c', u, v]\; X[n, c', i\cdot s + u, j\cdot s + v]$$

The same filter is applied at every spatial location — **parameter sharing**.

**Max pooling** takes the maximum within each $k\times k$ window; **average pooling** takes
the mean. Both reduce spatial resolution and provide a degree of translation tolerance.

We implement forward and backward via `im2col` / `col2im` (unrolling patches into columns
for efficient matrix multiply, then scattering gradients back).

In [2]:
def im2col(x, kH, kW, stride, pad):
    N, C, H, W = x.shape
    x_pad = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')
    H_out = (H + 2 * pad - kH) // stride + 1
    W_out = (W + 2 * pad - kW) // stride + 1
    cols = np.zeros((N, C, kH, kW, H_out, W_out), dtype=x.dtype)
    for kh in range(kH):
        for kw in range(kW):
            cols[:, :, kh, kw, :, :] = x_pad[:, :, kh:kh + stride * H_out:stride,
                                              kw:kw + stride * W_out:stride]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H_out * W_out, C * kH * kW), H_out, W_out


def col2im(cols, x_shape, kH, kW, stride, pad):
    N, C, H, W = x_shape
    H_out = (H + 2 * pad - kH) // stride + 1
    W_out = (W + 2 * pad - kW) // stride + 1
    cols = cols.reshape(N, H_out, W_out, C, kH, kW).transpose(0, 3, 4, 5, 1, 2)
    x_pad = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for kh in range(kH):
        for kw in range(kW):
            x_pad[:, :, kh:kh + stride * H_out:stride, kw:kw + stride * W_out:stride] += cols[:, :, kh, kw, :, :]
    return x_pad[:, :, pad:pad + H, pad:pad + W]


class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, seed=0):
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.in_channels, self.out_channels = in_channels, out_channels
        self.kH, self.kW = kernel_size
        self.stride, self.padding = stride, padding
        rng = np.random.RandomState(seed)
        scale = np.sqrt(2.0 / (in_channels * self.kH * self.kW))
        self.W = rng.randn(out_channels, in_channels, self.kH, self.kW) * scale
        self.b = np.zeros(out_channels)
        self.cache = None

    def forward(self, x):
        N, C, H, W = x.shape
        cols, H_out, W_out = im2col(x, self.kH, self.kW, self.stride, self.padding)
        W_col = self.W.reshape(self.out_channels, -1)
        out = (cols @ W_col.T + self.b).reshape(N, H_out, W_out, self.out_channels).transpose(0, 3, 1, 2)
        self.cache = (x, cols)
        return out

    def backward(self, dout):
        x, cols = self.cache
        dout_r = dout.transpose(0, 2, 3, 1).reshape(-1, self.out_channels)
        dW = (dout_r.T @ cols).reshape(self.W.shape)
        db = dout_r.sum(axis=0)
        dcols = dout_r @ self.W.reshape(self.out_channels, -1)
        dx = col2im(dcols, x.shape, self.kH, self.kW, self.stride, self.padding)
        return dx, dW, db


class MaxPool2d:
    def __init__(self, kernel_size, stride=None):
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kH, self.kW = kernel_size
        self.stride = self.kH if stride is None else stride
        self.cache = None

    def forward(self, x):
        N, C, H, W = x.shape
        cols, H_out, W_out = im2col(x, self.kH, self.kW, self.stride, 0)
        cols = cols.reshape(N, H_out, W_out, C, self.kH * self.kW)
        out = cols.max(axis=4)
        self.cache = (x.shape, cols.argmax(axis=4))
        return out.transpose(0, 3, 1, 2)

    def backward(self, dout):
        x_shape, argmax = self.cache
        N, C, H, W = x_shape
        H_out = (H - self.kH) // self.stride + 1
        W_out = (W - self.kW) // self.stride + 1
        dx = np.zeros(x_shape, dtype=dout.dtype)
        dout_r = dout.transpose(0, 2, 3, 1)
        for n in range(N):
            for c in range(C):
                for oh in range(H_out):
                    for ow in range(W_out):
                        idx = argmax[n, oh, ow, c]
                        kh, kw = divmod(idx, self.kW)
                        dx[n, c, oh * self.stride + kh, ow * self.stride + kw] += dout_r[n, oh, ow, c]
        return dx


class AvgPool2d:
    def __init__(self, kernel_size, stride=None):
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kH, self.kW = kernel_size
        self.stride = self.kH if stride is None else stride
        self.cache = None

    def forward(self, x):
        N, C, H, W = x.shape
        cols, H_out, W_out = im2col(x, self.kH, self.kW, self.stride, 0)
        cols = cols.reshape(N, H_out, W_out, C, self.kH * self.kW)
        out = cols.mean(axis=4)
        self.cache = x.shape
        return out.transpose(0, 3, 1, 2)

    def backward(self, dout):
        N, C, H, W = self.cache
        H_out = (H - self.kH) // self.stride + 1
        W_out = (W - self.kW) // self.stride + 1
        scale = 1.0 / (self.kH * self.kW)
        dx = np.zeros((N, C, H, W), dtype=dout.dtype)
        dout_r = dout.transpose(0, 2, 3, 1)
        for n in range(N):
            for c in range(C):
                for oh in range(H_out):
                    for ow in range(W_out):
                        for kh in range(self.kH):
                            for kw in range(self.kW):
                                dx[n, c, oh * self.stride + kh, ow * self.stride + kw] += dout_r[n, oh, ow, c] * scale
        return dx

print('Conv2d, MaxPool2d, and AvgPool2d defined.')

Conv2d, MaxPool2d, and AvgPool2d defined.


## 2. Validation Against PyTorch

Forward pass and analytical backward compared to `nn.Conv2d`, `nn.MaxPool2d`, and
`nn.AvgPool2d` in float64.

In [3]:
x_np = np.random.RandomState(42).randn(2, 3, 8, 8).astype(np.float64)
conv = Conv2d(3, 4, 3, padding=1, seed=42)
rng42 = np.random.RandomState(42)
conv.W = rng42.randn(4, 3, 3, 3).astype(np.float64) * 0.1
conv.b = rng42.randn(4).astype(np.float64)

out = conv.forward(x_np)
dout = rng42.randn(*out.shape).astype(np.float64)
dx, dW, db = conv.backward(dout)

x_t = torch.tensor(x_np, requires_grad=True)
layer = nn.Conv2d(3, 4, 3, padding=1, bias=True).double()
layer.weight.data = torch.tensor(conv.W)
layer.bias.data = torch.tensor(conv.b)
out_t = layer(x_t)
out_t.backward(torch.tensor(dout))
print(f"Conv forward max diff: {np.abs(out - out_t.detach().numpy()).max():.2e}")
print(f"Conv dx max diff:      {np.abs(dx - x_t.grad.numpy()).max():.2e}")
print(f"Conv dW max diff:      {np.abs(dW - layer.weight.grad.numpy()).max():.2e}")
print(f"Conv db max diff:      {np.abs(db - layer.bias.grad.numpy()).max():.2e}")

pool = MaxPool2d(2, 2)
pout = pool.forward(out)
dp = rng42.randn(*pout.shape).astype(np.float64)
pdx = pool.backward(dp)
x2 = torch.tensor(out, requires_grad=True)
pt = nn.MaxPool2d(2, 2).double()(x2)
pt.backward(torch.tensor(dp))
print(f"\nMaxPool dx max diff:   {np.abs(pdx - x2.grad.numpy()).max():.2e}")

apool = AvgPool2d(2, 2)
apout = apool.forward(out)
adp = rng42.randn(*apout.shape).astype(np.float64)
apdx = apool.backward(adp)
x3 = torch.tensor(out, requires_grad=True)
apt = nn.AvgPool2d(2, 2).double()(x3)
apt.backward(torch.tensor(adp))
print(f"AvgPool dx max diff:   {np.abs(apdx - x3.grad.numpy()).max():.2e}")

Conv forward max diff: 0.00e+00
Conv dx max diff:      1.11e-16
Conv dW max diff:      1.78e-14
Conv db max diff:      1.42e-14

MaxPool dx max diff:   0.00e+00
AvgPool dx max diff:   0.00e+00


## 3. Output Shapes & Edge-Detection Feature Maps

Output spatial size after conv or pool:
$$H_{out} = \lfloor (H + 2P - K) / S \rfloor + 1$$

A single conv layer with fixed Sobel filters produces interpretable **feature maps** —
horizontal and vertical edge responses — without any training.

In [4]:
H, W, K, P, S = 8, 8, 3, 1, 1
H_out = (H + 2 * P - K) // S + 1
print(f"8×8 input, 3×3 conv, pad=1, stride=1 → output {H_out}×{H_out}")
H2 = (H_out - 2) // 2 + 1
print(f"After 2×2 max-pool stride 2 → output {H2}×{H2}")

digits = load_digits()
img = digits.images[0].astype(np.float64) / 16.0
x_img = img[np.newaxis, np.newaxis]

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)
edge_conv = Conv2d(1, 2, 3, padding=1, seed=0)
edge_conv.W[0, 0] = sobel_x
edge_conv.W[1, 0] = sobel_y
edge_conv.b[:] = 0
feat = edge_conv.forward(x_img)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title(f"Digit '{digits.target[0]}'"); axes[0].axis('off')
axes[1].imshow(feat[0, 0], cmap='RdBu_r'); axes[1].set_title('Sobel-X (vertical edges)'); axes[1].axis('off')
axes[2].imshow(feat[0, 1], cmap='RdBu_r'); axes[2].set_title('Sobel-Y (horizontal edges)'); axes[2].axis('off')
plt.tight_layout()
plt.savefig('edge_feature_maps.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Sobel-X mean response: {feat[0, 0].mean():.3f}, Sobel-Y mean: {feat[0, 1].mean():.3f}")

8×8 input, 3×3 conv, pad=1, stride=1 → output 8×8
After 2×2 max-pool stride 2 → output 4×4
Sobel-X mean response: 0.000, Sobel-Y mean: 0.004


C:\Users\PM_IntellicaBD\AppData\Local\Temp\ipykernel_18912\1771184727.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Parameter Sharing

A 3×3 conv layer with 8 output channels on 8×8 images uses the **same 8 filters at every
spatial location**. A position-specific dense layer would need a separate 3×3 filter at
each of the 64 output positions — 8× more parameters per channel.

In [5]:
conv_params = 1 * 8 * 3 * 3 + 8  # 8 filters, 1 input channel, 3×3 kernel + bias
dense_per_filter = 8 * 8 * (3 * 3)  # unique 3×3 filter at each of 64 positions
print(f"Conv2d (8 filters, shared):     {conv_params} parameters")
print(f"Position-specific dense (×8):   {dense_per_filter * 8} parameters")
print(f"Sharing ratio:                    {(dense_per_filter * 8) / conv_params:.1f}× fewer params")

Conv2d (8 filters, shared):     80 parameters
Position-specific dense (×8):   4608 parameters
Sharing ratio:                    57.6× fewer params


## 5. Simple CNN vs Flattened MLP on Digits

We train a tiny CNN (Conv→ReLU→Pool ×2 → linear) and a flattened MLP (64→64→10) on
sklearn digits (8×8 grayscale, 10 classes). Same train/test split, 40 epochs, batch size 32.

In [6]:
def relu(x): return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(x.dtype)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class SimpleCNN:
    def __init__(self, seed=0):
        self.conv1 = Conv2d(1, 8, 3, padding=1, seed=seed)
        self.conv2 = Conv2d(8, 16, 3, padding=1, seed=seed + 1)
        self.pool1 = MaxPool2d(2, 2)
        self.pool2 = MaxPool2d(2, 2)
        rng = np.random.RandomState(seed + 2)
        self.Wfc = rng.randn(16 * 2 * 2, 10) * np.sqrt(2.0 / 64)
        self.bfc = np.zeros(10)

    def forward(self, x):
        c1 = self.conv1.forward(x)
        a1 = relu(c1)
        p1 = self.pool1.forward(a1)
        c2 = self.conv2.forward(p1)
        a2 = relu(c2)
        p2 = self.pool2.forward(a2)
        flat = p2.reshape(p2.shape[0], -1)
        return flat @ self.Wfc + self.bfc

    def step(self, x, y, lr=0.5):
        c1 = self.conv1.forward(x)
        a1 = relu(c1)
        p1 = self.pool1.forward(a1)
        c2 = self.conv2.forward(p1)
        a2 = relu(c2)
        p2 = self.pool2.forward(a2)
        flat = p2.reshape(p2.shape[0], -1)
        logits = flat @ self.Wfc + self.bfc
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy(); dlogits[np.arange(n), y] -= 1; dlogits /= n
        dWfc = flat.T @ dlogits; dbfc = dlogits.sum(axis=0)
        dflat = dlogits @ self.Wfc.T
        dp2 = dflat.reshape(p2.shape)
        da2 = self.pool2.backward(dp2)
        dc2 = da2 * relu_grad(c2)
        dp1, dW2, db2 = self.conv2.backward(dc2)
        da1 = self.pool1.backward(dp1)
        dc1 = da1 * relu_grad(c1)
        _, dW1, db1 = self.conv1.backward(dc1)
        self.Wfc -= lr * dWfc; self.bfc -= lr * dbfc
        self.conv2.W -= lr * dW2; self.conv2.b -= lr * db2
        self.conv1.W -= lr * dW1; self.conv1.b -= lr * db1
        return loss

    def predict(self, x):
        return self.forward(x).argmax(axis=1)

    def n_params(self):
        return self.conv1.W.size + self.conv2.W.size + self.Wfc.size


class MLPFlat:
    def __init__(self, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(64, 64) * np.sqrt(2.0 / 64)
        self.b1 = np.zeros(64)
        self.W2 = rng.randn(64, 10) * np.sqrt(2.0 / 64)
        self.b2 = np.zeros(10)

    def forward(self, x):
        self.z1 = x @ self.W1 + self.b1
        self.a1 = relu(self.z1)
        return self.a1 @ self.W2 + self.b2

    def step(self, x, y, lr=0.1):
        logits = self.forward(x)
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy(); dlogits[np.arange(n), y] -= 1; dlogits /= n
        dW2 = self.a1.T @ dlogits; db2 = dlogits.sum(axis=0)
        da1 = dlogits @ self.W2.T
        dz1 = da1 * relu_grad(self.z1)
        dW1 = x.T @ dz1; db1 = dz1.sum(axis=0)
        self.W2 -= lr * dW2; self.b2 -= lr * db2
        self.W1 -= lr * dW1; self.b1 -= lr * db1
        return loss

    def predict(self, x):
        return self.forward(x).argmax(axis=1)

    def n_params(self):
        return self.W1.size + self.W2.size


digits = load_digits()
X = digits.images.astype(np.float64) / 16.0
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
X_train_c = X_train[:, np.newaxis, :, :]
X_test_c = X_test[:, np.newaxis, :, :]
X_train_f = X_train.reshape(len(X_train), -1)
X_test_f = X_test.reshape(len(X_test), -1)

train_rng = np.random.RandomState(0)

cnn = SimpleCNN(seed=0)
for epoch in range(40):
    idx = train_rng.permutation(len(y_train))
    for i in range(0, len(idx), 32):
        b = idx[i:i + 32]
        cnn.step(X_train_c[b], y_train[b], lr=0.2)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>2} (CNN): test acc={(cnn.predict(X_test_c) == y_test).mean():.3f}")

train_rng = np.random.RandomState(0)
mlp = MLPFlat(seed=0)
for epoch in range(40):
    idx = train_rng.permutation(len(y_train))
    for i in range(0, len(idx), 32):
        b = idx[i:i + 32]
        mlp.step(X_train_f[b], y_train[b], lr=0.1)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>2} (MLP): test acc={(mlp.predict(X_test_f) == y_test).mean():.3f}")

print(f"\nFinal CNN test acc: { (cnn.predict(X_test_c) == y_test).mean():.3f}")
print(f"Final MLP test acc: { (mlp.predict(X_test_f) == y_test).mean():.3f}")
print(f"CNN parameters: {cnn.n_params()}  |  MLP parameters: {mlp.n_params()}")

Epoch 10 (CNN): test acc=0.947


Epoch 20 (CNN): test acc=0.973


Epoch 30 (CNN): test acc=0.973


Epoch 40 (CNN): test acc=0.976
Epoch 10 (MLP): test acc=0.956
Epoch 20 (MLP): test acc=0.969
Epoch 30 (MLP): test acc=0.973
Epoch 40 (MLP): test acc=0.973

Final CNN test acc: 0.976
Final MLP test acc: 0.973
CNN parameters: 1864  |  MLP parameters: 4736


## 6. Translation Robustness

Because conv filters are applied everywhere, a CNN should degrade more gracefully when
test digits are shifted by a few pixels. A flattened MLP treats each pixel index as a
separate input feature — shifting breaks the learned mapping.

In [7]:
def shift_images(X, dy, dx):
    out = np.zeros_like(X)
    H, W = X.shape[1], X.shape[2]
    ys, ye = max(0, dy), min(H, H + dy)
    xs, xe = max(0, dx), min(W, W + dx)
    yd, xd = max(0, -dy), max(0, -dx)
    out[:, ys:ye, xs:xe] = X[:, yd:yd + (ye - ys), xd:xd + (xe - xs)]
    return out

base_cnn = (cnn.predict(X_test_c) == y_test).mean()
base_mlp = (mlp.predict(X_test_f) == y_test).mean()
print(f"Baseline (no shift): CNN={base_cnn:.3f}, MLP={base_mlp:.3f}\n")
print(f"{'shift':>8} {'CNN':>8} {'MLP':>8}")
for dy, dx in [(0, 1), (0, 2), (1, 1), (2, 0)]:
    Xs = shift_images(X_test, dy, dx)
    acc_c = (cnn.predict(Xs[:, np.newaxis]) == y_test).mean()
    acc_m = (mlp.predict(Xs.reshape(len(Xs), -1)) == y_test).mean()
    print(f"({dy:+d},{dx:+d})  {acc_c:8.3f} {acc_m:8.3f}")

Baseline (no shift): CNN=0.976, MLP=0.973

   shift      CNN      MLP
(+0,+1)     0.656    0.411
(+0,+2)     0.404    0.109
(+1,+1)     0.184    0.147
(+2,+0)     0.122    0.116
